# EDGE AI CHALLENGE 2026 — Detection + Intent TextCNN

Notebook huấn luyện hai mô hình riêng biệt và tạo **một** `submission.csv`:
1. Object Detection: YOLOv8 custom siêu nhỏ, 7 lớp theo `classes.txt`.
2. Intent: một mô hình TextCNN hai nhánh (word + character), 5 nhãn theo thể lệ.

**Tuân thủ thể lệ:** không đọc/khai thác nhãn test, không pseudo-label test, không ensemble/TTA; dự đoán test đúng một lần/ảnh; submission giữ nguyên ID và thứ tự mẫu. Detector được huấn luyện từ đầu, không dùng pretrained ngoài. Kiểm tra giới hạn 200.000 tham số cho detection. Cần lượng tử hóa INT8 và kiểm tra operator/TFLite Micro riêng trước chung kết.

## 1. Cài đặt thư viện

In [ ]:
# Nếu Ultralytics đã có sẵn trong môi trường thì có thể bỏ qua cell này.
%pip install -q ultralytics


## 2. Import và cấu hình

In [ ]:
import os
import re
import json
import csv
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


## 3. Tìm và khai báo dataset
Chạy cell kế tiếp để xem thư mục được Kaggle gắn vào. Sau đó sửa `INPUT_DIR` nếu cần.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
print("Datasets trong /kaggle/input:")
for p in sorted(INPUT_ROOT.iterdir()):
    print(" -", p)

# Tự dò cấu trúc dataset. Nếu có nhiều dataset, có thể gán INPUT_DIR thủ công.
def find_dataset_root(root):
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for d in candidates:
        required = [
            d / "train" / "images",
            d / "train" / "labels",
            d / "test" / "images",
            d / "classes.txt",
            d / "intent_train.csv",
            d / "intent_test.csv",
            d / "sample_submission.csv",
        ]
        if all(x.exists() for x in required):
            return d
    return None

AUTO_INPUT_DIR = find_dataset_root(INPUT_ROOT)
if AUTO_INPUT_DIR is None:
    raise FileNotFoundError(
        "Không tự tìm thấy thư mục dữ liệu đủ cấu trúc. "
        "Hãy kiểm tra /kaggle/input và đặt INPUT_DIR = Path('đường_dẫn_dataset')."
    )

INPUT_DIR = AUTO_INPUT_DIR
print("Dataset được chọn:", INPUT_DIR)

WORK_DIR = Path("/kaggle/working")
YOLO_DATA_DIR = WORK_DIR / "yolo_dataset"
RUNS_DIR = WORK_DIR / "yolo_runs"
OUT_DIR = WORK_DIR / "outputs"
CKPT_DIR = WORK_DIR / "checkpoints"
for d in [OUT_DIR, CKPT_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_IMAGE_DIR = INPUT_DIR / "train" / "images"
TRAIN_LABEL_DIR = INPUT_DIR / "train" / "labels"
TEST_IMAGE_DIR = INPUT_DIR / "test" / "images"
CLASSES_FILE = INPUT_DIR / "classes.txt"
INTENT_TRAIN_FILE = INPUT_DIR / "intent_train.csv"
INTENT_TEST_FILE = INPUT_DIR / "intent_test.csv"
SAMPLE_SUBMISSION_FILE = INPUT_DIR / "sample_submission.csv"

required_paths = [
    TRAIN_IMAGE_DIR, TRAIN_LABEL_DIR, TEST_IMAGE_DIR,
    CLASSES_FILE, INTENT_TRAIN_FILE, INTENT_TEST_FILE,
    SAMPLE_SUBMISSION_FILE
]
for path in required_paths:
    assert path.exists(), f"Thiếu đường dẫn: {path}"
print("Đã xác nhận toàn bộ đường dẫn dữ liệu.")

## 4. Đọc danh sách lớp và kiểm tra dữ liệu

In [ ]:
with open(CLASSES_FILE, "r", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

NUM_CLASSES = len(classes)
print("Số lớp:", NUM_CLASSES)
for i, name in enumerate(classes):
    print(i, name)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
train_images = sorted(
    p for p in TRAIN_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
)
test_images = sorted(
    p for p in TEST_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
)

print(f"Train images: {len(train_images)}")
print(f"Test images : {len(test_images)}")
print(f"Train labels: {len(list(TRAIN_LABEL_DIR.glob('*.txt')))}")

missing_labels = [
    p.name for p in train_images
    if not (TRAIN_LABEL_DIR / f"{p.stem}.txt").exists()
]
print("Ảnh thiếu file label:", len(missing_labels), missing_labels[:10])


## 5. Kiểm tra nhãn YOLO và trực quan hóa
Định dạng nhãn phải là `class_id cx cy w h`, tọa độ chuẩn hóa 0–1. File `.txt` rỗng là ảnh nền; file thiếu cần được điều tra.

In [ ]:
def read_yolo_label(label_path):
    rows = []
    if not label_path.exists():
        return rows
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return rows
    for line_no, line in enumerate(text.splitlines(), start=1):
        vals = line.split()
        if len(vals) != 5:
            raise ValueError(f"{label_path}, dòng {line_no}: cần 5 giá trị")
        cls, cx, cy, w, h = map(float, vals)
        if not (0 <= int(cls) < NUM_CLASSES):
            raise ValueError(f"class_id ngoài phạm vi trong {label_path}: {cls}")
        if not all(0 <= v <= 1 for v in [cx, cy, w, h]):
            raise ValueError(f"Tọa độ YOLO ngoài [0,1] trong {label_path}")
        rows.append((int(cls), cx, cy, w, h))
    return rows

counter = Counter()
empty_count = 0
for p in train_images:
    lp = TRAIN_LABEL_DIR / f"{p.stem}.txt"
    if not lp.exists():
        continue
    labels = read_yolo_label(lp)
    if not labels:
        empty_count += 1
    counter.update(x[0] for x in labels)

print("Ảnh có label rỗng:", empty_count)
print("Số box theo lớp:")
for i, name in enumerate(classes):
    print(f"{i:2d} {name:35s} {counter[i]}")

def show_yolo_image(image_path):
    image_path = Path(image_path)
    im = Image.open(image_path).convert("RGB")
    W, H = im.size
    draw = ImageDraw.Draw(im)
    labels = read_yolo_label(TRAIN_LABEL_DIR / f"{image_path.stem}.txt")
    for cls, cx, cy, w, h in labels:
        x1 = (cx - w/2) * W
        y1 = (cy - h/2) * H
        x2 = (cx + w/2) * W
        y2 = (cy + h/2) * H
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1-14)), f"{cls}: {classes[cls]}", fill="yellow")
    plt.figure(figsize=(8, 6))
    plt.imshow(im)
    plt.axis("off")
    plt.show()

for p in train_images[:5]:
    show_yolo_image(p)


## 6. Chia train/validation và chuyển sang cấu trúc YOLO

Ultralytics cần file YAML và thư mục ảnh/nhãn tương ứng. Cell này tạo bản sao trong `/kaggle/working/yolo_dataset`; dữ liệu gốc ở `/kaggle/input` không bị thay đổi.

Để tránh lỗi do thiếu file nhãn, cell sẽ **dừng** nếu có ảnh không có file `.txt`. Nếu BTC xác nhận ảnh đó là ảnh nền, hãy tạo file nhãn rỗng tương ứng trong bản sao.


In [ ]:
# Chia validation có stratify theo lớp xuất hiện nhiều nhất trong mỗi ảnh.
# Đây chỉ là chia train/validation từ tập train được phép; không đọc nhãn test.
if missing_labels:
    raise FileNotFoundError(
        f"Có {len(missing_labels)} ảnh thiếu file nhãn. "
        "Chỉ tạo label rỗng nếu BTC xác nhận đó là ảnh nền."
    )

image_primary_labels = []
for img_path in train_images:
    rows = read_yolo_label(TRAIN_LABEL_DIR / f"{img_path.stem}.txt")
    image_primary_labels.append(
        Counter([r[0] for r in rows]).most_common(1)[0][0] if rows else NUM_CLASSES
    )

label_counts = Counter(image_primary_labels)
can_stratify = (
    len(label_counts) > 1
    and min(label_counts.values()) >= 2
    and int(round(len(train_images) * 0.2)) >= len(label_counts)
)
if can_stratify:
    train_paths, val_paths = train_test_split(
        train_images, test_size=0.2, random_state=SEED,
        shuffle=True, stratify=image_primary_labels
    )
else:
    print("Cảnh báo: không đủ mẫu để stratify; dùng split ngẫu nhiên cố định.")
    train_paths, val_paths = train_test_split(
        train_images, test_size=0.2, random_state=SEED, shuffle=True
    )

# Dọn dữ liệu sinh ra từ lần chạy cũ để tránh trộn split.
if YOLO_DATA_DIR.exists():
    shutil.rmtree(YOLO_DATA_DIR)
for split in ["train", "val"]:
    (YOLO_DATA_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def copy_yolo_split(paths, split):
    for img_path in paths:
        label_path = TRAIN_LABEL_DIR / f"{img_path.stem}.txt"
        shutil.copy2(img_path, YOLO_DATA_DIR / "images" / split / img_path.name)
        shutil.copy2(label_path, YOLO_DATA_DIR / "labels" / split / label_path.name)

copy_yolo_split(train_paths, "train")
copy_yolo_split(val_paths, "val")

yaml_path = YOLO_DATA_DIR / "data.yaml"
names_yaml = "\n".join(
    [f"  {i}: {json.dumps(name, ensure_ascii=False)}" for i, name in enumerate(classes)]
)
yaml_path.write_text(
    f"path: {YOLO_DATA_DIR}\ntrain: images/train\nval: images/val\nnames:\n{names_yaml}\n",
    encoding="utf-8"
)
print("Train:", len(train_paths), "| Validation:", len(val_paths))
print("Phân bố class chính toàn bộ:", dict(label_counts))

## 7. Huấn luyện YOLO nhỏ gọn

Dùng kiến trúc YOLOv8 custom với `width_multiple=0.0625`, `depth_multiple=0.10`, huấn luyện từ đầu để không vượt giới hạn tham số. Không sử dụng pretrained, ensemble hoặc TTA. Kết quả thực tế phải được xác nhận qua `mAP50` validation.

In [ ]:
# Detector YOLOv8 siêu nhỏ, huấn luyện từ đầu.
# depth_multiple/width_multiple được thu nhỏ để đáp ứng ngưỡng tham số.
# Không dùng ensemble hoặc test-time augmentation.
TINY_YOLO_YAML = WORK_DIR / "tiny_yolov8_edge.yaml"
tiny_yaml = f"""
nc: {NUM_CLASSES}
depth_multiple: 0.10
width_multiple: 0.02

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, true]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, true]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, true]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, true]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512, false]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256, false]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512, false]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024, false]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""
TINY_YOLO_YAML.write_text(tiny_yaml, encoding="utf-8")

MAX_MODEL_PARAMS = 200_000
YOLO_EPOCHS = 250
YOLO_IMGSZ = 160
YOLO_BATCH = 32 if torch.cuda.is_available() else 8

detector = YOLO(str(TINY_YOLO_YAML))
parameter_count = sum(p.numel() for p in detector.model.parameters())
print(f"YOLO parameters: {parameter_count:,} / {MAX_MODEL_PARAMS:,}")
if parameter_count > MAX_MODEL_PARAMS:
    raise ValueError(
        "Kiến trúc hiện tại vượt 200.000 tham số. "
        "Không được bỏ qua giới hạn; giảm width_multiple rồi kiểm tra lại."
    )

train_result = detector.train(
    data=str(yaml_path),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    patience=45,
    pretrained=False,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=5,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    seed=SEED,
    deterministic=True,
    project=str(RUNS_DIR),
    name="tiny_yolov8_edge",
    exist_ok=True,
    plots=True,
    verbose=True,
    # Augmentation vừa phải, hạn chế biến dạng làm sai nhãn hộp.
    hsv_h=0.015, hsv_s=0.45, hsv_v=0.30,
    degrees=5.0, translate=0.08, scale=0.25,
    shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=0.5, close_mosaic=20,
    mixup=0.0, copy_paste=0.0,
    amp=True
)

BEST_PT = RUNS_DIR / "tiny_yolov8_edge" / "weights" / "best.pt"
LAST_PT = RUNS_DIR / "tiny_yolov8_edge" / "weights" / "last.pt"
assert BEST_PT.exists(), f"Không tìm thấy checkpoint: {BEST_PT}"
print("Best checkpoint:", BEST_PT)

## 8. Đánh giá YOLO trên validation

In [ ]:
best_detector = YOLO(str(BEST_PT))
val_metrics = best_detector.val(
    data=str(yaml_path),
    imgsz=YOLO_IMGSZ,
    device=0 if torch.cuda.is_available() else "cpu",
    split="val",
    plots=True
)
print("Validation metrics:", val_metrics.results_dict)

## 9. Dự đoán tập test bằng YOLO

In [ ]:
# Chỉ một lần suy luận/ảnh; không TTA, không ensemble.
# conf thấp để không loại sớm box; NMS IoU được giữ ở mức vừa phải.
DET_CONF = 0.001
DET_IOU = 0.60
MAX_DETECTIONS = 100

test_results = best_detector.predict(
    source=[str(p) for p in test_images],
    imgsz=YOLO_IMGSZ,
    conf=DET_CONF,
    iou=DET_IOU,
    max_det=MAX_DETECTIONS,
    augment=False,
    save=False,
    verbose=False,
    device=0 if torch.cuda.is_available() else "cpu"
)

detection_predictions = {}
original_sizes = {}
for result in test_results:
    image_path = Path(result.path)
    sample_id = image_path.stem
    with Image.open(image_path) as im:
        orig_w, orig_h = im.size
    original_sizes[sample_id] = (orig_w, orig_h)

    preds = []
    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls.item())
            score = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1 = max(0, min(orig_w, int(round(x1))))
            y1 = max(0, min(orig_h, int(round(y1))))
            x2 = max(0, min(orig_w, int(round(x2))))
            y2 = max(0, min(orig_h, int(round(y2))))
            if x2 <= x1 or y2 <= y1:
                continue
            preds.append({
                "class_id": cls_id,
                "score": round(float(np.clip(score, 0.0, 1.0)), 6),
                "bbox": [x1, y1, x2, y2]
            })
    detection_predictions[sample_id] = preds

print("Đã dự đoán:", len(detection_predictions), "/", len(test_images), "ảnh")

## 10. Chuẩn bị dữ liệu Intent

Intent bắt buộc dùng mô hình học máy TextCNN, không thay bằng luật từ khóa. Chuẩn hóa dấu tiếng Việt về dạng không dấu để giảm khác biệt giữa câu hỏi có dấu/không dấu. Mô hình gồm nhánh từ và nhánh ký tự, vẫn là một TextCNN duy nhất.

In [ ]:
intent_train = pd.read_csv(INTENT_TRAIN_FILE)
intent_test = pd.read_csv(INTENT_TEST_FILE)

required_train_cols = {"question_id", "question", "intent"}
required_test_cols = {"question_id", "question"}
assert required_train_cols.issubset(intent_train.columns), intent_train.columns
assert required_test_cols.issubset(intent_test.columns), intent_test.columns

INTENT_LABELS = ["COUNT", "EXIST", "LIST", "LOCATION", "OUT_OF_SCOPE"]
intent_train = intent_train.dropna(subset=["question", "intent"]).copy()
intent_test["question"] = intent_test["question"].fillna("").astype(str)
intent_train["intent"] = intent_train["intent"].astype(str).str.strip().str.upper()
unknown_labels = set(intent_train["intent"].unique()) - set(INTENT_LABELS)
assert not unknown_labels, f"Nhãn không hợp lệ: {unknown_labels}"
assert intent_train["question_id"].is_unique
assert intent_test["question_id"].is_unique

def normalize_text(text):
    # Chuẩn hóa accent để câu có dấu/không dấu gần nhau hơn.
    text = unicodedata.normalize("NFD", str(text).lower().strip())
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = text.replace("đ", "d")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

intent_train["question_clean"] = intent_train["question"].map(normalize_text)
intent_test["question_clean"] = intent_test["question"].map(normalize_text)

print("Train:", len(intent_train), "| Test:", len(intent_test))
print("Phân bố intent:\n", intent_train["intent"].value_counts())
display(intent_train.head())

## 11. Chia train/validation và tạo vocabulary

In [ ]:
# Validation stratified theo intent để mỗi lớp đều được đánh giá.
min_class_count = intent_train["intent"].value_counts().min()
if min_class_count < 2:
    raise ValueError(
        "Có intent chỉ có 1 mẫu; cần thêm dữ liệu train có nhãn hoặc dùng "
        "cross-validation phù hợp, không thể tạo validation stratified đáng tin cậy."
    )

intent_tr_df, intent_val_df = train_test_split(
    intent_train, test_size=0.20, random_state=SEED,
    stratify=intent_train["intent"]
)

PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
label_to_id = {label: i for i, label in enumerate(INTENT_LABELS)}
id_to_label = {i: label for label, i in label_to_id.items()}

# Vocabulary chỉ học từ câu hỏi thuộc tập train chính thức (không dùng test).
# Dùng cả word token và character token cho biến thể gõ có/không dấu.
word_counts = Counter()
for s in intent_train["question_clean"]:
    word_counts.update(s.split())

word_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, _ in word_counts.most_common():
    word_vocab[token] = len(word_vocab)

char_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for ch in "abcdefghijklmnopqrstuvwxyz0123456789 ":
    if ch not in char_vocab:
        char_vocab[ch] = len(char_vocab)

MAX_WORD_LEN = min(40, max(12, max(len(s.split()) for s in intent_train["question_clean"]) + 2))
MAX_CHAR_LEN = min(120, max(40, max(len(s) for s in intent_train["question_clean"]) + 4))

def encode_word(text):
    ids = [word_vocab.get(t, word_vocab[UNK_TOKEN]) for t in text.split()[:MAX_WORD_LEN]]
    return ids + [0] * (MAX_WORD_LEN - len(ids))

def encode_char(text):
    ids = [char_vocab.get(ch, char_vocab[UNK_TOKEN]) for ch in text[:MAX_CHAR_LEN]]
    return ids + [0] * (MAX_CHAR_LEN - len(ids))

class IntentDataset(Dataset):
    def __init__(self, df, labeled=True):
        self.texts = df["question_clean"].astype(str).tolist()
        self.labeled = labeled
        self.labels = [label_to_id[x] for x in df["intent"]] if labeled else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = torch.tensor(encode_word(self.texts[idx]), dtype=torch.long)
        chars = torch.tensor(encode_char(self.texts[idx]), dtype=torch.long)
        if self.labeled:
            return words, chars, torch.tensor(self.labels[idx], dtype=torch.long)
        return words, chars

BATCH_SIZE = 16
train_loader = DataLoader(
    IntentDataset(intent_tr_df, True), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    IntentDataset(intent_val_df, True), batch_size=BATCH_SIZE, shuffle=False
)
print("Word vocab:", len(word_vocab), "| Char vocab:", len(char_vocab))
print("Train/val:", len(intent_tr_df), len(intent_val_df))

## 12. Định nghĩa TextCNN

In [ ]:
class TextCNN(nn.Module):
    """
    TextCNN hai nhánh: word-level + character-level.
    Vẫn là một mô hình TextCNN duy nhất, không dùng luật từ khóa để dự đoán intent.
    """
    def __init__(self, word_vocab_size, char_vocab_size, num_classes=5,
                 embedding_dim=32, filters=48, dropout=0.35):
        super().__init__()
        self.word_embedding = nn.Embedding(word_vocab_size, embedding_dim, padding_idx=0)
        self.char_embedding = nn.Embedding(char_vocab_size, 16, padding_idx=0)

        self.word_convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, filters, k) for k in (2, 3, 4)
        ])
        self.char_convs = nn.ModuleList([
            nn.Conv1d(16, 24, k) for k in (3, 4, 5)
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(filters * 3 + 24 * 3, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    @staticmethod
    def pool_features(x, convs):
        outputs = []
        for conv in convs:
            z = torch.relu(conv(x))
            outputs.append(torch.max(z, dim=2).values)
        return torch.cat(outputs, dim=1)

    def forward(self, word_ids, char_ids):
        w = self.word_embedding(word_ids).transpose(1, 2)
        c = self.char_embedding(char_ids).transpose(1, 2)
        wf = self.pool_features(w, self.word_convs)
        cf = self.pool_features(c, self.char_convs)
        return self.fc(self.dropout(torch.cat([wf, cf], dim=1)))

textcnn = TextCNN(
    word_vocab_size=len(word_vocab),
    char_vocab_size=len(char_vocab),
    num_classes=len(INTENT_LABELS)
).to(DEVICE)

n_params = sum(p.numel() for p in textcnn.parameters())
print(textcnn)
print(f"TextCNN parameters: {n_params:,}")

## 13. Huấn luyện TextCNN và lưu checkpoint tốt nhất

In [ ]:
# Huấn luyện có early stopping theo Accuracy validation (metric của cuộc thi).
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizer = torch.optim.AdamW(textcnn.parameters(), lr=8e-4, weight_decay=1e-4)
EPOCHS, PATIENCE = 180, 25
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=6, min_lr=1e-5
)

best_val_acc = -1.0
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    textcnn.train()
    losses = []
    for wb, cb, yb in train_loader:
        wb, cb, yb = wb.to(DEVICE), cb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = textcnn(wb, cb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(textcnn.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())

    textcnn.eval()
    val_true, val_pred = [], []
    with torch.no_grad():
        for wb, cb, yb in val_loader:
            logits = textcnn(wb.to(DEVICE), cb.to(DEVICE))
            val_pred.extend(logits.argmax(1).cpu().tolist())
            val_true.extend(yb.tolist())

    val_acc = accuracy_score(val_true, val_pred)
    scheduler.step(val_acc)
    train_loss = float(np.mean(losses))
    history.append({"epoch": epoch, "train_loss": train_loss, "val_accuracy": val_acc})
    print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in textcnn.state_dict().items()}
        torch.save({
            "model_state_dict": best_state,
            "word_vocab": word_vocab,
            "char_vocab": char_vocab,
            "max_word_len": MAX_WORD_LEN,
            "max_char_len": MAX_CHAR_LEN,
            "label_to_id": label_to_id,
            "classes": INTENT_LABELS
        }, CKPT_DIR / "best_textcnn.pth")
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

if best_state is not None:
    textcnn.load_state_dict(best_state)
pd.DataFrame(history).to_csv(OUT_DIR / "textcnn_training_history.csv", index=False)
print("Best validation accuracy:", best_val_acc)

## 14. Đánh giá TextCNN trên validation

In [ ]:
textcnn.eval()
val_true, val_pred = [], []
with torch.no_grad():
    for wb, cb, yb in val_loader:
        logits = textcnn(wb.to(DEVICE), cb.to(DEVICE))
        val_pred.extend(logits.argmax(1).cpu().tolist())
        val_true.extend(yb.tolist())

print("Validation accuracy:", accuracy_score(val_true, val_pred))
print(classification_report(
    val_true, val_pred, labels=list(range(len(INTENT_LABELS))),
    target_names=INTENT_LABELS, zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(val_true, val_pred, labels=list(range(len(INTENT_LABELS)))))

# Tạo vocabulary trên toàn bộ train có nhãn để tận dụng từ xuất hiện trong validation.
# Sau khi chốt số epoch từ validation, huấn luyện lại MỘT TextCNN trên toàn bộ train.
all_word_counts = Counter()
for s in intent_train["question_clean"]:
    all_word_counts.update(s.split())
final_word_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, _ in all_word_counts.most_common():
    final_word_vocab[token] = len(final_word_vocab)

def final_encode_word(text):
    ids = [final_word_vocab.get(t, 1) for t in text.split()[:MAX_WORD_LEN]]
    return ids + [0] * (MAX_WORD_LEN - len(ids))

class FinalIntentDataset(Dataset):
    def __init__(self, df):
        self.texts = df["question_clean"].astype(str).tolist()
        self.labels = [label_to_id[x] for x in df["intent"]]
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        s = self.texts[idx]
        w = torch.tensor(final_encode_word(s), dtype=torch.long)
        c = torch.tensor(encode_char(s), dtype=torch.long)
        return w, c, torch.tensor(self.labels[idx], dtype=torch.long)

final_textcnn = TextCNN(
    word_vocab_size=len(final_word_vocab),
    char_vocab_size=len(char_vocab),
    num_classes=len(INTENT_LABELS)
).to(DEVICE)
full_loader = DataLoader(
    FinalIntentDataset(intent_train), batch_size=BATCH_SIZE, shuffle=True
)
final_optimizer = torch.optim.AdamW(final_textcnn.parameters(), lr=8e-4, weight_decay=1e-4)
final_epochs = max(20, len(history))
for epoch in range(final_epochs):
    final_textcnn.train()
    for wb, cb, yb in full_loader:
        wb, cb, yb = wb.to(DEVICE), cb.to(DEVICE), yb.to(DEVICE)
        final_optimizer.zero_grad(set_to_none=True)
        loss = criterion(final_textcnn(wb, cb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_textcnn.parameters(), 1.0)
        final_optimizer.step()

textcnn = final_textcnn
word_vocab = final_word_vocab
torch.save({
    "model_state_dict": textcnn.state_dict(),
    "word_vocab": word_vocab,
    "char_vocab": char_vocab,
    "max_word_len": MAX_WORD_LEN,
    "max_char_len": MAX_CHAR_LEN,
    "label_to_id": label_to_id,
    "classes": INTENT_LABELS
}, CKPT_DIR / "final_textcnn_full_train.pth")
print("Final TextCNN trained on all labeled train rows:", len(intent_train))

## 15. Dự đoán intent_test.csv

In [ ]:
class TestIntentDataset(Dataset):
    def __init__(self, df):
        self.texts = df["question_clean"].astype(str).tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        s = self.texts[idx]
        wb = torch.tensor(encode_word(s), dtype=torch.long)
        cb = torch.tensor(encode_char(s), dtype=torch.long)
        return wb, cb

test_intent_loader = DataLoader(
    TestIntentDataset(intent_test), batch_size=32, shuffle=False
)
intent_predictions = []
textcnn.eval()
with torch.no_grad():
    for wb, cb in test_intent_loader:
        logits = textcnn(wb.to(DEVICE), cb.to(DEVICE))
        pred_ids = logits.argmax(dim=1).cpu().tolist()
        intent_predictions.extend(id_to_label[i] for i in pred_ids)

assert len(intent_predictions) == len(intent_test)
intent_prediction_dict = dict(zip(
    intent_test["question_id"].astype(str), intent_predictions
))
intent_pred_df = intent_test[["question_id", "question"]].copy()
intent_pred_df["prediction"] = intent_predictions
intent_pred_df.to_csv(OUT_DIR / "intent_predictions.csv", index=False)
display(intent_pred_df.head())

## 16. Tạo submission.csv

In [ ]:
# Tạo ánh xạ dự đoán theo ID trước, sau đó đi theo đúng thứ tự của sample_submission.csv.
prediction_lookup = {}

for image_path in test_images:
    sample_id = str(image_path.stem)
    preds = detection_predictions.get(image_path.stem, [])
    prediction_lookup[("DETECTION", sample_id)] = json.dumps(
        preds, ensure_ascii=False, separators=(",", ":")
    )

for _, row in intent_test.iterrows():
    qid = str(row["question_id"])
    prediction_lookup[("INTENT", qid)] = str(intent_prediction_dict[qid])

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_FILE, dtype=str).fillna("")
expected_columns = ["task_type", "sample_id", "prediction"]
if not set(expected_columns).issubset(sample_submission.columns):
    raise ValueError(
        "sample_submission.csv cần có các cột "
        f"{expected_columns}; hiện có {sample_submission.columns.tolist()}"
    )

# Giữ nguyên toàn bộ thứ tự, ID và các cột trong file mẫu.
submission = sample_submission.copy()
missing_keys = []
for idx, row in submission.iterrows():
    key = (str(row["task_type"]).strip().upper(), str(row["sample_id"]).strip())
    if key not in prediction_lookup:
        missing_keys.append(key)
    else:
        submission.at[idx, "prediction"] = prediction_lookup[key]

if missing_keys:
    raise ValueError(
        "Không tạo được dự đoán cho một số ID trong sample_submission.csv. "
        f"Ví dụ: {missing_keys[:10]}"
    )

submission_path = WORK_DIR / "submission.csv"
submission.to_csv(
    submission_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL
)

print("Saved:", submission_path)
print("Rows:", len(submission))
display(submission.head(10))


## 17. Kiểm tra định dạng submission

In [ ]:
check = pd.read_csv(submission_path, dtype=str).fillna("")
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_FILE, dtype=str).fillna("")

assert check.columns.tolist() == sample_submission.columns.tolist()
assert len(check) == len(sample_submission)
assert check["sample_id"].tolist() == sample_submission["sample_id"].tolist()
assert check["task_type"].tolist() == sample_submission["task_type"].tolist()
assert not check["sample_id"].duplicated().any()
assert set(check["task_type"].str.upper().unique()).issubset({"DETECTION", "INTENT"})

det_rows = check[check["task_type"].str.upper() == "DETECTION"]
int_rows = check[check["task_type"].str.upper() == "INTENT"]

assert set(det_rows["sample_id"]) == {p.stem for p in test_images}
assert set(int_rows["sample_id"]) == set(intent_test["question_id"].astype(str))
assert set(int_rows["prediction"]).issubset(set(INTENT_LABELS))

for _, row in det_rows.iterrows():
    preds = json.loads(row["prediction"])
    assert isinstance(preds, list) and len(preds) <= 100
    image_size = original_sizes[row["sample_id"]]
    W, H = image_size
    for obj in preds:
        assert set(obj.keys()) == {"class_id", "score", "bbox"}
        cls_id = int(obj["class_id"])
        score = float(obj["score"])
        bbox = obj["bbox"]
        assert 0 <= cls_id < NUM_CLASSES
        assert 0 <= score <= 1
        assert isinstance(bbox, list) and len(bbox) == 4
        assert all(isinstance(v, int) for v in bbox)
        x1, y1, x2, y2 = bbox
        assert 0 <= x1 < x2 <= W
        assert 0 <= y1 < y2 <= H

print("PASS: định dạng, ID và thứ tự khớp sample_submission.csv.")
print("DETECTION:", len(det_rows), "| INTENT:", len(int_rows))
print("Submission:", submission_path)


## 18. Kiểm tra và lưu kết quả

Notebook tạo `/kaggle/working/submission.csv`, `best.pt`, checkpoint TextCNN và lịch sử train.

### Các giới hạn cần xác minh trước khi nộp/chung kết
- Detection: không quá 200.000 tham số; ảnh đầu vào 160×160; tối đa 100 box/ảnh.
- Intent: một TextCNN; cần chuyển sang TFLite INT8, kiểm tra kích thước dưới 1,5 MB và toán tử TFLite Micro hỗ trợ.
- Detection cũng cần lượng tử hóa INT8, kiểm tra kích thước dưới 1,5 MB và khả năng chạy trên ESP32-S3.
- Notebook không dùng pretrained hoặc dữ liệu ngoài. Nếu bổ sung dữ liệu/trọng số pretrained sau này, phải khai báo chính xác nguồn theo thể lệ.
- Điểm validation không bảo đảm điểm private. Chỉ dùng validation thuộc train để chọn cấu hình; tuyệt đối không dò nhãn test, pseudo-label test hoặc ghi cứng đáp án.
- Mỗi đội chỉ được nộp tối đa 4 lần trong 24 giờ; chọn bài nộp chính thức theo quy định BTC.